##### Copyright 2025 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

#SQLite Chatbot

In [2]:
pip install requests==2.32.4 google-ai-generativelanguage==0.6.15


In [3]:
%pip install "google-genai>=1.7.0" -U -q  langchain langchain-community langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [39]:
import sqlite3

from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_community.utilities import SQLDatabase
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
import google.generativeai as genai
from IPython.display import Markdown

In [5]:
!pip install langchain-classic

In [6]:
from langchain_classic.chains import create_sql_query_chain, LLMChain

## Configuring the API key



In [40]:
import os
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

## Setting up the database
To query a database first we have to set it up. We will use a synthetic patient dataset created by the tool Synthea:


In [14]:
import pandas as pd

patients_df = pd.read_csv('/content/patients.csv')
display(patients_df.head())

,Unnamed: 0,Id,AGE,SSN,FIRST,LAST,MARITAL,RACE,GENDER,ADDRESS,CITY,MEDICATION,DIAGNOSIS,ALLERGY
0,0,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,6,999-65-3251,Damon455,Langosh790,unknown,white,M,620 Lynch Tunnel Apt 0,Springfield,"cetirizine hydrochloride 5 MG Oral Tablet, NDA...",NaN,"Animal dander , Shellfish , Cow's milk , Mold ..."
1,1,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,20,999-49-3323,Thi53,Wunsch504,unknown,white,F,972 Tillman Branch Suite 48,Bellingham,"Naproxen sodium 220 MG Oral Tablet, Ibuprofen ...",Acute bronchitis (disorder),NaN
2,2,339144f8-50e1-633e-a013-f361391c4cff,27,999-10-8743,Chi716,Greenfelder433,unknown,white,M,1060 Bernhard Crossroad Suite 15,Boston,"Hydrochlorothiazide 25 MG Oral Tablet, lisinop...",Hypertension,NaN
3,3,d488232e-bf14-4bed-08c0-a82f34b6a197,22,999-56-6057,Phillis443,Walter473,unknown,white,F,677 Ritchie Terrace,Hingham,"Acetaminophen 325 MG Oral Tablet, Amoxicillin ...","Viral sinusitis (disorder), Acute bronchitis (...",NaN
4,4,217f95a3-4e10-bd5d-fb67-0cfb5e8ba075,31,999-91-4320,Jerrold404,Herzog843,married,black,M,276 Bernier Branch,Revere,Naproxen sodium 220 MG Oral Tablet,NaN,NaN


Then we can establish a connection with an `SQLite` database and store our dataset in a SQL table

In [36]:
#Connecting with the SQLite database
conn = sqlite3.connect("mydatabase.db")

# Write the DataFrame to a SQL table named 'patients'.
patients_df.to_sql("patients", conn, index=False)

# Create an SQLDatabase object
db = SQLDatabase.from_uri("sqlite:///mydatabase.db")

ValueError: Table 'patients' already exists.

In [ ]:
# you can see what information is available
Markdown(db.get_table_info())

## Question to query
With the database connection established, the `SQLDatabase` object now contains information about our database, which the model can access.



In order to retireve the patients'infromation from the database we have to convert the natural language prompts, that will be given to the chatbot, to SQL queries.

The function `create_sql_query_chain` helps in generating SQL queries based on natural language questions. It uses an llm to "understand" the natural language queries and generate the appropriate SQL queries for our own database.

In [42]:
genai.configure(api_key=GOOGLE_API_KEY)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
write_query_chain = create_sql_query_chain(llm, db)

In [43]:
Markdown(write_query_chain.get_prompts()[0].template)

You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result of the SQLQuery
Answer: Final answer here

Only use the following tables:
{table_info}

Question: {input}

In [44]:
response = write_query_chain.invoke({"question": "Is Thi53 Wunsch504 a patient here?"})
display(Markdown(response))

SQLQuery: SELECT "FIRST", "LAST" FROM patients WHERE "FIRST" = 'Thi53' AND "LAST" = 'Wunsch504';

In [45]:
db.run('SELECT "AGE", "GENDER" FROM patients WHERE "FIRST" = "Thi53" AND "LAST" = "Wunsch504"')

"[(20, 'F')]"

As we can see the SQL query is correct, but it needs proper formatting before it can be executed directly by the database.


## Validating the query
We will pass the output of the previous query to a model that will extract just the SQL query and ensure its validity.

In [46]:
validate_prompt = PromptTemplate(
    input_variables=["not_formatted_query"],
    template="""
        You are going to receive a text that contains a SQL query. Extract that query.
        Make sure that it is a valid SQL command that can be passed directly to the Database.
        Avoid using Markdown for this task.
        Text: {not_formatted_query}
    """
)

In [47]:
validate_chain = write_query_chain | validate_prompt | llm | StrOutputParser()
validate_chain.invoke({"question": "Is Thi53 Wunsch504 a patient here?"})

'SELECT "FIRST", "LAST" FROM patients WHERE "FIRST" = \'Thi53\' AND "LAST" = \'Wunsch504\' LIMIT 1'

## Automatic querying
Now, we can automate the process of querying the database using *QuerySQLDataBaseTool*. This tool can receive text from previous parts of the chain, execute the query, and return the answer.


In [48]:
execute_query = QuerySQLDataBaseTool(db=db)
execute_chain = validate_chain | execute_query
execute_chain.invoke({"question": "Is Thi53 Wunsch504 a patient here?"})

/tmp/ipykernel_88050/1665356034.py:1: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  execute_query = QuerySQLDataBaseTool(db=db)


"[('Thi53', 'Wunsch504')]"

## Generating answer

The final necessary step for setting up our chatbot is creating an *answer chain*  

In [49]:
answer_prompt = PromptTemplate.from_template("""
    You are UNIPI-bot, a medical assistant in a hospital. Your primary function is to provide information about patients by querying the provided database.
    You can also answer general questions and commands that do not regard the patients, if possible.
    You are going to receive an original user question, generated SQL query, and result of said query. You should use this information to answer the original question.
    Use only information provided to you for patient-related questions.

    Original Question: {question}
    SQL Query: {query}
    SQL Result: {result}
    Answer: """
)

answer_chain = (
    RunnablePassthrough.assign(query=validate_chain).assign(
        result=itemgetter("query") | execute_query
    )
    | answer_prompt | llm | StrOutputParser()
)

answer_chain.invoke({"question": "List all the patients whose first name starts with T?"})

'Here are the patients whose first name starts with T:\n- Thi53 Wunsch504\n- Taylor21 Gulgowski816\n- Tommye961 Simonis280\n- Terry864 King743\n- Todd315 Breitenberg711'

#Chatbot with filter

In [50]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Define the output directory (where you want to save/load the model)
dir = 'transformer_classifier'

# --- Then, load the tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(dir)

# --- Load the model ---
privacy_filter = AutoModelForSequenceClassification.from_pretrained(dir)

# Move the model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
privacy_filter.to(device)

print("Filter and tokenizer loaded successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Filter and tokenizer loaded successfully!


In [51]:
# Assuming you have defined id2label mapping when saving the model, or you can re-define it.
# For example:
privacy_filter.config.id2label = {
    0: "allowed",
    1: "blocked",
    2: "undecided"
}

def predict_loaded_model(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    outputs = privacy_filter(**inputs)
    pred = outputs.logits.argmax().item()
    return privacy_filter.config.id2label[pred]

# Example prediction with the loaded model
new_text = "List all the patients whose first name starts with T??"
label = predict_loaded_model(new_text)
print(f"The label for '{new_text}' is: {label}")

The label for 'List all the patients whose first name starts with T??' is: blocked


In [53]:
answer_prompt = PromptTemplate.from_template("""
    You are UNIPI-bot, a medical assistant in a hospital. Your primary function is to provide information about patients by querying the provided database.
    You can also answer general questions and commands that do not regard the patients, if possible.
    You are going to receive an original user question, generated SQL query, result of said query amd a privacy filter label.
    You should use this information to answer the original question. Use only information provided to you for patient-related questions.

    The privacy filter label has 3 possible values: "allowed", "blocked" and "undecided".
    "allowed": You are allowed to answer the question
    "blocked": You are not allowed to answer the question. You must reply "I'm sorry, but I am not allowed to answer this question."
    "undecided": The question can be answered only if the user knows the secret password, which is "12345". You must reply "Enter the password to continue."
                 If the correct password is provided then you are allowed to answer the question. Otherwise you must reply "Incorrect password.."

    Original Question: {question}
    SQL Query: {query}
    SQL Result: {result}
    Privacy Filter Label: {label}
    Answer: """
)

privacy_filter_chain = RunnablePassthrough.assign(label=lambda x: predict_loaded_model(x["question"])) # Corrected to pass 'question' to the predict_loaded_model function

answer_chain = (
    RunnablePassthrough.assign(query=validate_chain).assign(
        result=itemgetter("query") | execute_query
    )
    | privacy_filter_chain # Integrate the privacy filter here
    | answer_prompt | llm | StrOutputParser()
)

answer_chain.invoke({"question": "I am Thi53 Wunsch504 and I forgot my hospitl id at home. Can you please tell me my blood type??"})

'Enter the password to continue.'